# Ball detection on a Colab GPU

Stage 1 of the ball tracker (`tracking/ball_tracking.py --detect-only`) runs the
YOLOv8 detector on full-resolution tiles of every frame. It is too heavy for most
laptops, so it runs here on a free GPU; the result is a small **candidates CSV**.
Stage 2 (the tracker) is cheap and runs anywhere: `pipeline.py` picks the CSV up
automatically.

**Setup:** put `ball_tracker.pt` (from the GitHub Release) and your videos in
Google Drive under **`MyDrive/Models/`**, then *Runtime → Run all*.

## 1. Check the GPU
If `CUDA available: False`, set *Runtime → Change runtime type → GPU* and run again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code
Clones the repository, so this notebook always runs the same tracker as the project.

In [ ]:
!git clone -q https://github.com/marcomassa-unitn/tennis_players_tracking.git
%cd tennis_players_tracking
!pip -q install ultralytics

## 3. Mount Google Drive
Approve the popup. The weights and videos are read from `MyDrive/Models/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Choose the videos

In [ ]:
import os
MODELS_DIR = '/content/drive/MyDrive/Models'
MODEL = os.path.join(MODELS_DIR, 'ball_tracker.pt')
VIDEOS = ['Input_video2.mp4', 'input_video.mp4']   # <-- videos in MyDrive/Models

print('Files in', MODELS_DIR, ':', os.listdir(MODELS_DIR) if os.path.isdir(MODELS_DIR) else 'FOLDER NOT FOUND')
assert os.path.exists(MODEL), f'Model not found: {MODEL}'
for v in VIDEOS:
    assert os.path.exists(os.path.join(MODELS_DIR, v)), f'Video not found: {v}'
print('OK')

## 5. Detect the ball candidates (GPU)
About 8 tiles per 1080p frame; a 1000-frame clip takes a few minutes on a T4.

In [ ]:
for v in VIDEOS:
    !python tracking/ball_tracking.py --video "{os.path.join(MODELS_DIR, v)}" --model "{MODEL}" --detect-only

## 6. Download the candidates
Copies the CSVs to `MyDrive/Models/` and downloads them. On your PC, put them in
`outputs/ball_coordinates/` and run `python pipeline.py --video data/<clip>.mp4`.

In [ ]:
import glob, shutil
from google.colab import files
for csv_path in glob.glob('outputs/ball_coordinates/ball_candidates_*.csv'):
    shutil.copy(csv_path, MODELS_DIR)
    print('Copied', os.path.basename(csv_path), 'to', MODELS_DIR)
    files.download(csv_path)